<a href="https://colab.research.google.com/github/Benjaminotuya1/ADNI-Cognitive-Survival-XGBoost/blob/main/XGBoost_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [5]:
# Step 1: Connect to the Data Source
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/ADNI_Survival_Project/data/'

print("1. Loading Data Pillars...")
adni_merge = pd.read_csv(base_path + 'ADNIMERGE_02Jun2026(1).csv', low_memory=False)
lab_data = pd.read_csv(base_path + 'LABDATA_04Jun2026.csv', low_memory=False)
crp_data = pd.read_csv(base_path + 'adni_plasma_qc_multiplex_11Nov2010.csv', low_memory=False)
hdl_data = pd.read_csv(base_path + 'ADNINIGHTINGALELONG_05_24_21_08Jun2026.csv', low_memory=False)

print("2. Normalizing Timecodes (The Engineering Fixes)...")
# FIX A: Pull the Screening basic bloodwork forward to match the Baseline MRI
lab_data['VISCODE'] = lab_data['VISCODE'].replace({'sc': 'bl'})

# FIX B: Intercept the rogue CRP column from the independent lab
crp_data = crp_data.rename(columns={'Visit_Code': 'VISCODE'})

print("3. Executing Master Merge...")
master_df = pd.merge(adni_merge, lab_data, on=['RID', 'VISCODE'], how='left')
master_df = pd.merge(master_df, crp_data, on=['RID', 'VISCODE'], how='left')
master_df = pd.merge(master_df, hdl_data, on=['RID', 'VISCODE'], how='left')

print("4. Cleaning and Filtering for Mod4NeuCog...")
keep_columns = [
    'RID', 'VISCODE', 'Years_bl', 'DX',
    'AGE', 'PTGENDER', 'APOE4',
    'Hippocampus', 'ICV', 'WholeBrain',
    'RCT11', 'RCT20', 'RCT19',
    'C-Reactive Protein (CRP) (ug/mL)',
    'HDL_C'
]

# Keep only the target columns
clean_df = master_df[keep_columns].copy()

# Translate obscure codes into plain English
clean_df = clean_df.rename(columns={
    'RCT11': 'Fasting_Glucose',
    'RCT20': 'Total_Cholesterol',
    'RCT19': 'Triglycerides',
    'C-Reactive Protein (CRP) (ug/mL)': 'CRP',
    'HDL_C': 'HDL'
})

# Drop visits without a clinical diagnosis
initial_rows = len(clean_df)
clean_df = clean_df.dropna(subset=['DX'])
dropped_rows = initial_rows - len(clean_df)

print(f"   -> Dropped {dropped_rows} visits due to missing Diagnosis.")
print(f"   -> Final Analysis Cohort: {len(clean_df)} patient visits.")

print("5. Saving Final Dataset...")
output_path = base_path + 'mod4neucog_final_dataset.csv'
clean_df.to_csv(output_path, index=False)
print(f"Success! Data locked and saved to: {output_path}")

# Let's instantly check if the missing values are fixed
print("\n--- NEW MISSING VALUES CHECK ---")
print(clean_df.isnull().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Loading Data Pillars...
2. Normalizing Timecodes (The Engineering Fixes)...
3. Executing Master Merge...
4. Cleaning and Filtering for Mod4NeuCog...
   -> Dropped 4963 visits due to missing Diagnosis.
   -> Final Analysis Cohort: 11478 patient visits.
5. Saving Final Dataset...
Success! Data locked and saved to: /content/drive/MyDrive/ADNI_Survival_Project/data/mod4neucog_final_dataset.csv

--- NEW MISSING VALUES CHECK ---
RID                      0
VISCODE                  0
Years_bl                 0
DX                       0
AGE                      7
PTGENDER                 0
APOE4                  282
Hippocampus           3322
ICV                   2152
WholeBrain            2495
Fasting_Glucose      10524
Total_Cholesterol    10524
Triglycerides        10524
CRP                  10409
HDL                  11478
dtype: int64


In [6]:
import pandas as pd
hdl_path = '/content/drive/MyDrive/ADNI_Survival_Project/data/ADNINIGHTINGALELONG_05_24_21_08Jun2026.csv'
hdl_df = pd.read_csv(hdl_path, low_memory=False)

print("--- HOW NIGHTINGALE SPELLS VISCODE ---")
print(hdl_df['VISCODE'].unique()[:15])

if 'VISCODE2' in hdl_df.columns:
    print("\n--- HOW NIGHTINGALE SPELLS VISCODE2 ---")
    print(hdl_df['VISCODE2'].unique()[:15])

--- HOW NIGHTINGALE SPELLS VISCODE ---
['v21' 'nv' 'v11' 'v06' 'v41' 'v31' 'v03' 'v05' 'v51']

--- HOW NIGHTINGALE SPELLS VISCODE2 ---
['m96' 'm60' 'm84' 'm72' 'bl' 'm12' 'm24' 'm120' 'm36' 'm18' 'm06' 'm108'
 'm48']


In [7]:
# Step 1: Connect to the Data Source
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/ADNI_Survival_Project/data/'

print("1. Loading Data Pillars...")
adni_merge = pd.read_csv(base_path + 'ADNIMERGE_02Jun2026(1).csv', low_memory=False)
lab_data = pd.read_csv(base_path + 'LABDATA_04Jun2026.csv', low_memory=False)
crp_data = pd.read_csv(base_path + 'adni_plasma_qc_multiplex_11Nov2010.csv', low_memory=False)
hdl_data = pd.read_csv(base_path + 'ADNINIGHTINGALELONG_05_24_21_08Jun2026.csv', low_memory=False)

print("2. Normalizing Timecodes...")
# FIX A: Pull Screening bloodwork forward to Baseline
lab_data['VISCODE'] = lab_data['VISCODE'].replace({'sc': 'bl'})

# FIX B: Rename the rogue CRP column
crp_data = crp_data.rename(columns={'Visit_Code': 'VISCODE'})

# FIX C: Overwrite Nightingale's messy timecodes with their clean ones
hdl_data['VISCODE'] = hdl_data['VISCODE2']

print("3. Executing Master Merge...")
master_df = pd.merge(adni_merge, lab_data, on=['RID', 'VISCODE'], how='left')
master_df = pd.merge(master_df, crp_data, on=['RID', 'VISCODE'], how='left')
master_df = pd.merge(master_df, hdl_data, on=['RID', 'VISCODE'], how='left')

print("4. Cleaning and Filtering...")
keep_columns = [
    'RID', 'VISCODE', 'Years_bl', 'DX',
    'AGE', 'PTGENDER', 'APOE4',
    'Hippocampus', 'ICV', 'WholeBrain',
    'RCT11', 'RCT20', 'RCT19',
    'C-Reactive Protein (CRP) (ug/mL)',
    'HDL_C'
]

clean_df = master_df[keep_columns].copy()

clean_df = clean_df.rename(columns={
    'RCT11': 'Fasting_Glucose',
    'RCT20': 'Total_Cholesterol',
    'RCT19': 'Triglycerides',
    'C-Reactive Protein (CRP) (ug/mL)': 'CRP',
    'HDL_C': 'HDL'
})

initial_rows = len(clean_df)
clean_df = clean_df.dropna(subset=['DX'])
dropped_rows = initial_rows - len(clean_df)

print(f"   -> Dropped {dropped_rows} visits due to missing Diagnosis.")
print(f"   -> Final Analysis Cohort: {len(clean_df)} patient visits.")

print("5. Saving Final Dataset...")
output_path = base_path + 'mod4neucog_final_dataset.csv'
clean_df.to_csv(output_path, index=False)
print(f"Success! Data locked and saved to: {output_path}")

print("\n--- FINAL MISSING VALUES CHECK ---")
print(clean_df.isnull().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Loading Data Pillars...
2. Normalizing Timecodes...
3. Executing Master Merge...
4. Cleaning and Filtering...
   -> Dropped 4963 visits due to missing Diagnosis.
   -> Final Analysis Cohort: 11518 patient visits.
5. Saving Final Dataset...
Success! Data locked and saved to: /content/drive/MyDrive/ADNI_Survival_Project/data/mod4neucog_final_dataset.csv

--- FINAL MISSING VALUES CHECK ---
RID                      0
VISCODE                  0
Years_bl                 0
DX                       0
AGE                      7
PTGENDER                 0
APOE4                  282
Hippocampus           3332
ICV                   2156
WholeBrain            2501
Fasting_Glucose      10564
Total_Cholesterol    10564
Triglycerides        10564
CRP                  10435
HDL                   6631
dtype: int64


In [8]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11518 entries, 0 to 16472
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RID                11518 non-null  int64  
 1   VISCODE            11518 non-null  object 
 2   Years_bl           11518 non-null  float64
 3   DX                 11518 non-null  object 
 4   AGE                11511 non-null  float64
 5   PTGENDER           11518 non-null  object 
 6   APOE4              11236 non-null  float64
 7   Hippocampus        8186 non-null   float64
 8   ICV                9362 non-null   float64
 9   WholeBrain         9017 non-null   float64
 10  Fasting_Glucose    954 non-null    object 
 11  Total_Cholesterol  954 non-null    object 
 12  Triglycerides      954 non-null    object 
 13  CRP                1083 non-null   float64
 14  HDL                4887 non-null   float64
dtypes: float64(8), int64(1), object(6)
memory usage: 1.4+ MB


In [10]:
print(clean_df['Fasting_Glucose'].dropna().unique()[:20])

['106' '96' '78' '94' '180' '89' '82' '172' '97' '95' '81' '110' '77'
 '149' '413' '88' '104' '76' '100' '117']


In [13]:
df = clean_df
print("--- BEFORE FIX ---")
print(df[['Fasting_Glucose', 'Total_Cholesterol', 'Triglycerides']].dtypes)

# Force the text columns into numeric decimals, safely turning errors into blanks
cols_to_fix = ['Fasting_Glucose', 'Total_Cholesterol', 'Triglycerides']

for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n--- AFTER FIX ---")
print(df[['Fasting_Glucose', 'Total_Cholesterol', 'Triglycerides']].dtypes)

--- BEFORE FIX ---
Fasting_Glucose      object
Total_Cholesterol    object
Triglycerides        object
dtype: object

--- AFTER FIX ---
Fasting_Glucose      float64
Total_Cholesterol    float64
Triglycerides        float64
dtype: object


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11518 entries, 0 to 16472
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RID                11518 non-null  int64  
 1   VISCODE            11518 non-null  object 
 2   Years_bl           11518 non-null  float64
 3   DX                 11518 non-null  object 
 4   AGE                11511 non-null  float64
 5   PTGENDER           11518 non-null  object 
 6   APOE4              11236 non-null  float64
 7   Hippocampus        8186 non-null   float64
 8   ICV                9362 non-null   float64
 9   WholeBrain         9017 non-null   float64
 10  Fasting_Glucose    940 non-null    float64
 11  Total_Cholesterol  943 non-null    float64
 12  Triglycerides      943 non-null    float64
 13  CRP                1083 non-null   float64
 14  HDL                4887 non-null   float64
dtypes: float64(11), int64(1), object(3)
memory usage: 1.4+ MB


In [15]:
import pandas as pd
import numpy as np

print("--- TABLE 1: AVERAGE PATIENT BIOLOGY BY DIAGNOSIS ---")

# We group the patients by their Diagnosis (DX), then calculate the mean for the biological markers
table_1 = df.groupby('DX')[
    ['AGE', 'Hippocampus', 'WholeBrain', 'Fasting_Glucose', 'CRP', 'HDL']
].mean().round(2)

# Display the clean table
display(table_1)

--- TABLE 1: AVERAGE PATIENT BIOLOGY BY DIAGNOSIS ---


,AGE,Hippocampus,WholeBrain,Fasting_Glucose,CRP,HDL
DX,,,,,,
CN,72.95,7320.47,1023858.02,101.58,0.33,1.56
Dementia,74.27,5584.65,962579.90,98.95,0.11,1.53
MCI,72.91,6774.60,1027490.04,102.16,0.11,1.53
